# Bagging for Player Churn Prediction

This notebook demonstrates **Bagging (Bootstrap Aggregating)**, an ensemble
learning technique, applied to a gaming-domain problem: predicting whether a
player will churn (stop playing) in the next 7 days.

We compare three models:
1. A single **Decision Tree** (baseline, high variance)
2. A **Bagging ensemble** of decision trees (bootstrap sampling + voting)
3. A **Random Forest** (bagging + random feature subsets per split)

> **Open in Colab:** Runtime → Run all. No file uploads needed — the dataset
> is generated synthetically so the notebook runs end to end on its own.

## 1. Install dependencies

Colab already ships with these, but running the install keeps the notebook
reproducible on any environment.

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib

## 2. Import libraries

Each group of imports gets its own cell so it's clear what each library is
used for.

**2.1 Data handling**

In [ ]:
import numpy as np
import pandas as pd

**2.2 Plotting**

In [ ]:
import matplotlib.pyplot as plt

**2.3 Train/test split**

In [ ]:
from sklearn.model_selection import train_test_split

**2.4 Models — baseline tree, bagging, random forest**

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier

**2.5 Evaluation metrics**

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

## 3. Create the synthetic gaming dataset

Features are typical telemetry a game backend already logs: session
frequency, session length, win rate, recent purchases, recency, and social
connectedness.

**3.1 Generate player features**

In [ ]:
rng = np.random.default_rng(42)
n_players = 2000

sessions_per_week = rng.poisson(lam=5, size=n_players)
avg_session_minutes = rng.normal(30, 12, n_players).clip(1)
win_rate = rng.beta(2, 2, n_players)
purchases_last_30d = rng.poisson(lam=1.5, size=n_players)
days_since_last_login = rng.exponential(scale=3, size=n_players).clip(0, 30)
friends_in_game = rng.poisson(lam=4, size=n_players)

**3.2 Derive the churn label**

The label is built from a noisy combination of the features above. The added
noise is what makes a single decision tree prone to overfitting — and is
exactly what bagging is designed to average out.

In [ ]:
churn_score = (
    -0.35 * sessions_per_week
    - 0.02 * avg_session_minutes
    - 1.2 * win_rate
    - 0.4 * purchases_last_30d
    + 0.25 * days_since_last_login
    - 0.15 * friends_in_game
    + rng.normal(0, 1.5, n_players)
)
churn = (churn_score > np.median(churn_score)).astype(int)

**3.3 Assemble the dataframe**

In [ ]:
df = pd.DataFrame({
    "sessions_per_week": sessions_per_week,
    "avg_session_minutes": avg_session_minutes,
    "win_rate": win_rate,
    "purchases_last_30d": purchases_last_30d,
    "days_since_last_login": days_since_last_login,
    "friends_in_game": friends_in_game,
    "churned": churn,
})
df.head()

## 4. Train/test split

25% held out for testing, stratified on the label so both sets keep the same
churn ratio.

In [ ]:
X = df.drop(columns="churned")
y = df["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
X_train.shape, X_test.shape

## 5. Baseline model — single decision tree

This is the model bagging is meant to improve on. Trained once, on the full
training set, with no ensembling.

**5.1 Train**

In [ ]:
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

**5.2 Test / evaluate**

In [ ]:
tree_pred = tree.predict(X_test)

print(f"Single tree accuracy: {accuracy_score(y_test, tree_pred):.3f}")
print(f"Single tree F1 score: {f1_score(y_test, tree_pred):.3f}")

## 6. Bagging ensemble

`BaggingClassifier` draws `n_estimators` bootstrap samples (random rows,
**with replacement**) from the training set, fits one base estimator per
sample, and aggregates predictions by majority vote.

**6.1 Train**

In [ ]:
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,     # number of bootstrap samples / trees
    max_samples=1.0,      # each sample is 100% the size of the training set (with replacement)
    bootstrap=True,
    n_jobs=-1,
    random_state=42,
)
bagging.fit(X_train, y_train)

**6.2 Test / evaluate**

In [ ]:
bagging_pred = bagging.predict(X_test)

print(f"Bagging accuracy: {accuracy_score(y_test, bagging_pred):.3f}")
print(f"Bagging F1 score: {f1_score(y_test, bagging_pred):.3f}")

## 7. Random Forest

Bagging plus one more trick: each split only considers a random subset of
features (`max_features="sqrt"`), which decorrelates the trees further and
usually improves on plain bagging.

**7.1 Train**

In [ ]:
forest = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42,
)
forest.fit(X_train, y_train)

**7.2 Test / evaluate**

In [ ]:
forest_pred = forest.predict(X_test)

print(f"Random forest accuracy: {accuracy_score(y_test, forest_pred):.3f}")
print(f"Random forest F1 score: {f1_score(y_test, forest_pred):.3f}")

## 8. Compare all three models

**8.1 Results table**

In [ ]:
results = pd.DataFrame({
    "model": ["Single decision tree", "Bagging (100 trees)", "Random forest (100 trees)"],
    "accuracy": [
        accuracy_score(y_test, tree_pred),
        accuracy_score(y_test, bagging_pred),
        accuracy_score(y_test, forest_pred),
    ],
    "f1_score": [
        f1_score(y_test, tree_pred),
        f1_score(y_test, bagging_pred),
        f1_score(y_test, forest_pred),
    ],
})
results

**8.2 Bar chart comparison**

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(results))
width = 0.35

ax.bar(x - width/2, results["accuracy"], width, label="Accuracy")
ax.bar(x + width/2, results["f1_score"], width, label="F1 score")
ax.set_xticks(x)
ax.set_xticklabels(results["model"], rotation=10, ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Single tree vs. bagging vs. random forest")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Score a new player

In production this is the part that matters: given one player's recent
stats, output a churn *probability* (not just a label) so it can feed a
retention campaign trigger.

In [ ]:
new_player = pd.DataFrame([{
    "sessions_per_week": 1,
    "avg_session_minutes": 8,
    "win_rate": 0.3,
    "purchases_last_30d": 0,
    "days_since_last_login": 12,
    "friends_in_game": 1,
}])

churn_probability = forest.predict_proba(new_player)[0, 1]
print(f"New player churn risk (Random Forest): {churn_probability:.1%}")

## 10. Feature importance

Bagged tree ensembles give a free bonus: averaged feature importance across
all trees, which is far more stable than reading importance off one tree.

In [ ]:
importances = pd.Series(forest.feature_importances_, index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
importances.plot.barh(ax=ax, color="#16C79A")
ax.set_title("Random forest feature importance")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

## Takeaways

- **Bagging trades a bit of interpretability for a large drop in variance** —
  the ensemble is far less sensitive to noisy training rows than a single tree.
- **Random Forest usually edges out plain bagging** because the extra
  feature-level randomness decorrelates the trees even further.
- In a gaming churn model, that stability matters: a single overfit tree
  might flag the wrong players, wasting retention-campaign budget. The
  ensemble's steadier probability estimates are what you'd actually wire
  into a live trigger.